# 01 - Exploratory Data Analysis

**Project** Football Betting Integrity Monitor

**Last updated:** 27.05.2026


## Overview

Steps of exploratory data analysis for German Bundesliga, English Premier League, Turkish and Greek top leagues for seasons 17/18 until 25/26.

## 1. Data inspection

In [6]:
# Inspect downloaded files

import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')

# Load all CSVs into a dict keyed by filename stem (e.g. 'D1_1718')
raw_files = sorted(RAW_DIR.glob('*.csv'))
print(f"Found {len(raw_files)} CSV files")

Found 36 CSV files


In [7]:
# Row counts per league and season

rows_data = []

for f in raw_files:
    league, season = f.stem.split('_')
    df = pd.read_csv(f)
    rows_data.append({'league': league, 'season': season, 'matches': len(df)})

rows_df = pd.DataFrame(rows_data)
pivot = rows_df.pivot(index='season', columns='league', values='matches')
print(pivot.to_string())

league   D1   E0   G1   T1
season                    
1718    306  380  240  306
1819    306  380  240  306
1920    306  380  240  306
2021    306  380  240  420
2122    306  380  240  380
2223    306  380  240  342
2324    306  380  240  380
2425    306  380  233  342
2526    306  380  236  306


In [8]:
# Column schema audit

schema_data = []

for f in raw_files:
    league, season = f.stem.split('_')
    df = pd.read_csv(f, nrows=1)
    schema_data.append({
        'league': league,
        'season': season,
        'n_columns': len(df.columns),
        'columns': list(df.columns)
    })

schema_df = pd.DataFrame(schema_data)

# Summary: column counts per league per season
pivot_schema = schema_df.pivot(index='season', columns='league', values='n_columns')
print("Column counts per league/season:")
print(pivot_schema.to_string())

# Flag any seasons that differ from the most common count per league
print("\nAnomalies (seasons with non-standard column count):")
for league in ['D1', 'E0', 'T1', 'G1']:
    subset = schema_df[schema_df['league'] == league]
    most_common = subset['n_columns'].mode()[0]
    anomalies = subset[subset['n_columns'] != most_common]
    if anomalies.empty:
        print(f"  {league}: all seasons consistent ({most_common} columns)")
    else:
        for _, row in anomalies.iterrows():
            print(f"  {league} {row['season']}: {row['n_columns']} columns (expected {most_common})")

Column counts per league/season:
league   D1   E0   G1   T1
season                    
1718     64   65   64   64
1819     61   62   61   61
1920    105  106  105  105
2021    105  106  105  105
2122    105  106  105  105
2223    105  106  105  105
2324    105  106  105  105
2425    119  120  119  119
2526    131  132  131  131

Anomalies (seasons with non-standard column count):
  D1 1718: 64 columns (expected 105)
  D1 1819: 61 columns (expected 105)
  D1 2425: 119 columns (expected 105)
  D1 2526: 131 columns (expected 105)
  E0 1718: 65 columns (expected 106)
  E0 1819: 62 columns (expected 106)
  E0 2425: 120 columns (expected 106)
  E0 2526: 132 columns (expected 106)
  T1 1718: 64 columns (expected 105)
  T1 1819: 61 columns (expected 105)
  T1 2425: 119 columns (expected 105)
  T1 2526: 131 columns (expected 105)
  G1 1718: 64 columns (expected 105)
  G1 1819: 61 columns (expected 105)
  G1 2425: 119 columns (expected 105)
  G1 2526: 131 columns (expected 105)


In [9]:
# Evaluating column consistency accross all files

CORE_COLS = [
    'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'B365H', 'B365D', 'B365A',
    'MaxH', 'MaxD', 'MaxA',
    'AvgH', 'AvgD', 'AvgA'
]

print("Core column availability across all files:\n")
missing_any = False

for f in sorted(raw_files):
    league, season = f.stem.split('_')
    df = pd.read_csv(f, nrows=1)
    missing = [c for c in CORE_COLS if c not in df.columns]
    if missing:
        print(f"  MISSING in {league}_{season}: {missing}")
        missing_any = True

if not missing_any:
    print("  All 15 core columns present in all 36 files ✓")

Core column availability across all files:

  MISSING in D1_1718: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in D1_1819: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in E0_1718: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in E0_1819: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in G1_1718: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in G1_1819: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in T1_1718: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
  MISSING in T1_1819: ['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']


### Note:
**As seasons 17/18 and 18/19 missing crucial data (max and average odds) we are not going to use them and only work with data for 7 seasons between 2019/2020 and 2025/2026**

## 2. Data cleaning and merging

In [ ]:
# Clean the data, keep only relevant columns and combine all data into one master dataframe

CORE_COLS = [
    'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'B365H', 'B365D', 'B365A',
    'MaxH', 'MaxD', 'MaxA',
    'AvgH', 'AvgD', 'AvgA'
]

TIER_MAP = {
    'D1': 'elite',
    'E0': 'elite',
    'T1': 'mid_tier',
    'G1': 'mid_tier'
}

SEASONS_TO_USE = ["1920", "2021", "2122", "2223", "2324", "2425", "2526"]

frames = []

for f in sorted(Path('../data/raw').glob('*.csv')):
    league, season = f.stem.split('_')

    if season not in SEASONS_TO_USE:
        continue

    df = pd.read_csv(f, usecols=CORE_COLS)
    df['league'] = league
    df['season'] = season
    df['tier']   = TIER_MAP[league]

    frames.append(df)

master = pd.concat(frames, ignore_index=True)

print(f"Total rows:    {len(master):,}")
print(f"Total columns: {len(master.columns)}")
print(f"\nLeague counts:\n{master['league'].value_counts()}")
print(f"\nTier counts:\n{master['tier'].value_counts()}")
print(f"\nSeason counts:\n{master['season'].value_counts().sort_index()}")

Total rows:    8,947
Total columns: 18

League counts:
league
E0    2660
T1    2476
D1    2142
G1    1669
Name: count, dtype: int64

Tier counts:
tier
elite       4802
mid_tier    4145
Name: count, dtype: int64

Season counts:
season
1920    1232
2021    1346
2122    1306
2223    1268
2324    1306
2425    1261
2526    1228
Name: count, dtype: int64


In [11]:
# Fix data types

# Fix date column
master['Date'] = pd.to_datetime(master['Date'], dayfirst=True)

# Ensure score columns are integer
master['FTHG'] = master['FTHG'].astype('Int64')
master['FTAG'] = master['FTAG'].astype('Int64')

# Ensure odds columns are float
odds_cols = ['B365H', 'B365D', 'B365A', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']
for col in odds_cols:
    master[col] = pd.to_numeric(master[col], errors='coerce')

# Check result
print(master.dtypes)
print(f"\nDate range: {master['Date'].min().date()} → {master['Date'].max().date()}")

Date        datetime64[us]
HomeTeam               str
AwayTeam               str
FTHG                 Int64
FTAG                 Int64
FTR                    str
B365H              float64
B365D              float64
B365A              float64
MaxH               float64
MaxD               float64
MaxA               float64
AvgH               float64
AvgD               float64
AvgA               float64
league                 str
season                 str
tier                   str
dtype: object

Date range: 2019-08-09 → 2026-05-24


In [14]:
# Double-check all column names

master.head(2)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,league,season,tier
0,2019-08-16,Bayern Munich,Hertha,2,2,D,1.14,8.0,15.0,1.25,8.30,17.5,1.18,7.55,15.04,D1,1920,elite
1,2019-08-17,Dortmund,Augsburg,5,1,H,1.20,7.0,13.0,1.25,7.15,17.0,1.22,6.62,13.38,D1,1920,elite


### Column decode

**Match Identity**

- **Date** -  DateDate the match was played

- **HomeTeam** - Team Name of the home team

- **AwayTeam** - Team Name of the away team

**Match Result**

- **FTHG** - Full Time Home Goals - Goals scored by home team
- **FTAG** - Full Time Away Goals - Goals scored by away team
- **FTR** -  Full Time Result - H = Home win, D = Draw, A = Away win

**Bet365 Odds**

- **B365H** - Bet365 Home Win Odds 
- **B365D** - Bet365 Draw Odds
- **B365A** - Bet365 Away Win Odds

**Market Maximum Odds**

- **MaxH** - Maximum Home Win Odds - Best available home win odds across all bookmakers
- **MaxD** - Maximum Draw Odds - Best available draw odds across all bookmakers
- **MaxA** - Maximum Away Win Odds - Best available away win odds across all bookmakers

**Market Average Odds**

- **AvgH** - Average Home Win Odds - Market average home win odds across all bookmakers
- **AvgD** - Average Draw Odds - Market average draw odds across all bookmakers
- **AvgA** - Average Away Win Odds - Market average away win odds across all bookmakers

**Metadata** (added by us)

- **league** - League - CodeD1 = Bundesliga, E0 = EPL, T1 = Turkey, G1 = Greece
- **season** - Season - Codee.g. 1920 = 2019/20 season
- **tier** - League Tier - elite = (Germany, England), mid_tier = (Turkey, Greece)

In [15]:
## Data quality audit

# 1. Null counts
print("=== Null counts ===")
nulls = master.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "No nulls found ✓")

# 2. Duplicate rows
dupes = master.duplicated().sum()
print(f"\n=== Duplicate rows: {dupes} ===")

# 3. FTR valid values
print(f"\n=== FTR values ===")
print(master['FTR'].value_counts())

# 4. Odds sanity — all odds should be > 1.0
print("\n=== Odds below 1.0 (invalid) ===")
for col in ['B365H', 'B365D', 'B365A', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA']:
    bad = (master[col] < 1.0).sum()
    if bad > 0:
        print(f"  {col}: {bad} invalid values")
print("All odds valid ✓") if all((master[col] >= 1.0).all() for col in ['B365H','B365D','B365A','MaxH','MaxD','MaxA','AvgH','AvgD','AvgA']) else None

# 5. Max >= Avg sanity check
print("\n=== Max < Avg anomalies (data errors) ===")
for outcome in ['H', 'D', 'A']:
    bad = (master[f'Max{outcome}'] < master[f'Avg{outcome}']).sum()
    if bad > 0:
        print(f"  Max{outcome} < Avg{outcome}: {bad} rows")
    else:
        print(f"  Max{outcome} >= Avg{outcome} always ✓")

=== Null counts ===
B365H    56
B365D    56
B365A    56
MaxH     30
MaxD     30
MaxA     30
AvgH     30
AvgD     30
AvgA     30
dtype: int64

=== Duplicate rows: 0 ===

=== FTR values ===
FTR
H    3884
A    2818
D    2245
Name: count, dtype: int64

=== Odds below 1.0 (invalid) ===

=== Max < Avg anomalies (data errors) ===
  MaxH >= AvgH always ✓
  MaxD >= AvgD always ✓
  MaxA >= AvgA always ✓


In [16]:
## Evaluating where the nulls are coming from

print("=== Where are the nulls? ===\n")

# B365 nulls by league and season
b365_nulls = master[master['B365H'].isnull()].groupby(['league','season']).size()
print("B365 nulls by league/season:")
print(b365_nulls)

# Max/Avg nulls by league and season
max_nulls = master[master['MaxH'].isnull()].groupby(['league','season']).size()
print("\nMax/Avg nulls by league/season:")
print(max_nulls)

=== Where are the nulls? ===

B365 nulls by league/season:
league  season
G1      1920       3
        2021       4
        2122       2
        2526       8
T1      2021       5
        2223      33
        2324       1
dtype: int64

Max/Avg nulls by league/season:
league  season
G1      2021       1
T1      2223      29
dtype: int64


In [17]:
## Checking if nulls in b365 odds are consistent with nulls in max odds

null_b365 = set(master[master['B365H'].isnull()].index)
null_max = set(master[master['MaxH'].isnull()].index)

print(f"Rows missing B365 only:        {len(null_b365 - null_max)}")
print(f"Rows missing Max/Avg only:      {len(null_max - null_b365)}")
print(f"Rows missing both:              {len(null_b365 & null_max)}")
print(f"Total rows with any null:       {len(null_b365 | null_max)}")

Rows missing B365 only:        26
Rows missing Max/Avg only:      0
Rows missing both:              30
Total rows with any null:       56


In [18]:
## Drop unrecoverable rows that can't be used for our analysis

# Drop rows where Max/Avg are null (no odds data at all - unusable)
before = len(master)
master = master.dropna(subset=['MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA'])
after = len(master)

print(f"Rows dropped: {before - after}")
print(f"Rows remaining: {after:,}")
print(f"\nRemaining nulls:")
print(master.isnull().sum()[master.isnull().sum() > 0])

Rows dropped: 30
Rows remaining: 8,917

Remaining nulls:
B365H    26
B365D    26
B365A    26
dtype: int64


## 3. Save the master dataset as CSV file

In [ ]:
## Save the master dataset as CSV file

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

master.to_csv(PROCESSED_DIR / 'master.csv', index=False)

print(f"Saved: data/processed/master.csv")
print(f"Shape: {master.shape}")
print(f"\nPreview:")
master.head()

Saved: data/processed/master.csv
Shape: (8917, 18)

Preview:


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,league,season,tier
0,2019-08-16,Bayern Munich,Hertha,2,2,D,1.14,8.00,15.00,1.25,8.30,17.50,1.18,7.55,15.04,D1,1920,elite
1,2019-08-17,Dortmund,Augsburg,5,1,H,1.20,7.00,13.00,1.25,7.15,17.00,1.22,6.62,13.38,D1,1920,elite
2,2019-08-17,Freiburg,Mainz,3,0,H,2.25,3.25,3.40,2.26,3.49,3.65,2.20,3.37,3.36,D1,1920,elite
3,2019-08-17,Leverkusen,Paderborn,3,2,H,1.25,6.00,12.00,1.31,6.40,12.25,1.28,5.97,10.27,D1,1920,elite
4,2019-08-17,Werder Bremen,Fortuna Dusseldorf,1,3,A,1.75,3.75,4.75,1.80,4.05,4.95,1.74,3.93,4.57,D1,1920,elite
